In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "hribar2011great")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Hribar call 2011_Experiment_1&2_RawData.sav")
complete_path_2 = os.path.join(original_data_pathway, "Hribar Call 2011_Experiment_3_RawData.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
df2 = pd.read_spss(complete_path_2, usecols=None, convert_categoricals=True)


df1['date'] = df1['date'].astype(str)
df1[['year', 'month', 'day']] = df1['date'].str.split('-',expand=True)
df2['experiment'] = "exp3"


In [3]:
data_frames=[df1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"name": "ape", 
        "species":"species_original",
        "date":"date_original"}, inplace=True)
    x['study_id']="hribar2011great"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

fulldf['experiment'].replace('exp', '', inplace=True, regex=True)


In [4]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')
# fulldf.columns
fulldf.rename(columns={"ape": "participant"}, inplace=True)

In [5]:
replace_spaces = ['condition', 'landmark', 'landmark2', 'filter_$']
for x in replace_spaces:
    fulldf[x].replace(' ', '_', inplace=True, regex=True)

replace_condition = [[1, 'full_table_near'], [2, 'half_table_near'], 
    [3, 'full_table_far'], [4, 'half_table_far']]
for x,y in replace_condition:
    fulldf['condition'].replace(x, y, inplace=True)

In [6]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc'].replace('nan-nan-nan', np.nan, inplace=True)
fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])
fulldf['dob'] = pd.to_datetime(fulldf['dob'])

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

In [7]:
fulldf=fulldf[['study_id', 'experiment', 
         'year', 'month', 'day',  'participant','age_in_years','sex',   'species',
        'session', 'trial',  'condition','position', 'chosen', 'correct',  'start_side', 'side',
       'table_part', 'table', 'distance_between_cups', 'distance_to_edge',
       'landmark', 'landmark2', 'chosen_landmark']]


In [8]:
for index in range(1,4):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'hribar2011great_standardized_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'hribar2011great_standardized_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)